In [1]:
import numpy as np
import torch
import tensorflow as tf

2024-04-21 11:55:37.786126: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-21 11:55:37.869485: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-21 11:55:37.869522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-21 11:55:37.883868: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-21 11:55:37.916118: I tensorflow/core/platform/cpu_feature_guar

In [2]:
tf.test.is_gpu_available()

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


2024-04-21 11:55:39.205992: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 11:55:39.207931: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 11:55:39.208000: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

True

L355
2024-04-21 11:55:39.283105: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 11:55:39.283160: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 11:55:39.283209: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 19847 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
PER_FRAME_FEATURE = 1629
def extractPoseHand(npy):
    np_matrix=npy.reshape(-1,PER_FRAME_FEATURE)

    pose  = np_matrix[: ,0:98]
    hands  = np_matrix[: ,1503:PER_FRAME_FEATURE]
    
 
    arr = np.concatenate((pose, hands), axis=1)


    return arr

In [4]:
def getNPY_224(split, name):
   file_name =  f"../relativeQ/BdSLW60_FLIPPED/{split}/{name}" 
    
   npy = np.load(file_name)
   npy = extractPoseHand(npy)
   return torch.from_numpy(npy)


In [5]:
import pickle
import gzip
import os

# dict_keys(['name', 'signer', 'gloss', 'text', 'sign'])

# Test
test_path  = "../relativeQ/BdSLW60_FLIPPED/Test"
annotations = []   

for trial in os.listdir(test_path):
    sample = {}

    sample['name'] = "Test/"+trial
    sample['signer'] = trial.split("_")[1]
    sample['gloss'] = trial.split("_")[0]
    sample['text'] = trial.split("_")[0]
    sample['sign'] = getNPY_224("Test", trial)

    annotations.append(sample)


In [6]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.test", "wb") as file:
    pickle.dump(annotations, file)
file.close()

In [7]:
VALIDATION_USER ="U3"

In [8]:
# Dev
train_path  = "../relativeQ/BdSLW60_FLIPPED/Training"
annotations_dev = []   
annotations_train = []   

for trial in os.listdir(train_path):
    sample = {}
    
    sample['signer'] = trial.split("_")[1]
    sample['gloss'] = trial.split("_")[0]
    sample['text'] = trial.split("_")[0]
    sample['sign'] = getNPY_224("Training", trial)

    user = trial.split("_")[1]
    if user == VALIDATION_USER:
        sample['name'] = "Train/"+trial
        annotations_dev.append(sample)
    else:
        sample['name'] = "Train/"+trial
        annotations_train.append(sample)

In [9]:
  
import random
train_path  = "../relativeQ/BdSLW60_FLIPPED/Training"
fileList = os.listdir(train_path)
fileList.sort()



p_list = []

all_train = []
all_dev = []

previous_trial = fileList[0]
s = previous_trial.split("_")
p_prefix = s[0]+"_"+s[1]



for index in range(1, len(fileList)):
   
    p_list.append(previous_trial)
    

    next_trial = fileList[index]
    s = next_trial.split("_")
    next_prefix = s[0]+"_"+s[1]

    if next_prefix != p_prefix or index == len(fileList)-1:
        if index == len(fileList)-1:
            p_list.append(next_trial)
        # do process
        if len(p_list) != 0:
           random.shuffle(p_list)

           size = len(p_list)
           dev_size = int(size * 0.1)

           dev_set = p_list[:dev_size]
           train_set = p_list[dev_size:]
           total_size = len(dev_set) + len(train_set)
           assert  total_size == size, f"{len(dev_set)}  {len(train_set)}  {size}"

           all_train.extend(train_set)
           all_dev.extend(dev_set)
           p_list = []

        # else:
        #     print("Empty list")

    previous_trial = next_trial
    p_prefix = next_prefix


In [10]:
# import pickle
# import gzip
# import os

# # Train

# annotations = []   
# for trial in all_train:
#     sample = {}

#     sample['name'] = "Train/"+trial
#     sample['signer'] = trial.split("_")[1]
#     sample['gloss'] = trial.split("_")[0]
#     sample['text'] = trial.split("_")[0]
#     sample['sign'] = getNPY_224("Training", trial)

#     annotations.append(sample)
# # save the gzipped file
# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.UBS.train", "wb") as file:
#     pickle.dump(annotations, file)
# file.close()

In [11]:
# import pickle
# import gzip
# import os

# # Dev

# annotations = []   
# for trial in all_dev:
#     sample = {}

#     sample['name'] = "Train/"+trial
#     sample['signer'] = trial.split("_")[1]
#     sample['gloss'] = trial.split("_")[0]
#     sample['text'] = trial.split("_")[0]
#     sample['sign'] = getNPY_224("Training", trial)

#     annotations.append(sample)
# # save the gzipped file
# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.UBS.dev", "wb") as file:
#     pickle.dump(annotations, file)
# file.close()

In [12]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.train", "wb") as file:
    pickle.dump(annotations_train, file)
file.close()

In [13]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.dev", "wb") as file:
    pickle.dump(annotations_dev, file)
file.close()

In [14]:
# # check sanity
# i = 0
# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.UBS.train", "rb") as file:
#     data_test = pickle.load(file)
#     print(len(data_test))

# file.close()

In [15]:
# # check sanity

# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.dev", "rb") as file:
#     data_test = pickle.load(file)
#     print(len(data_test))

# file.close()

In [16]:
# check sanity
i = 0
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.FLIP.test", "rb") as file:
    data_test = pickle.load(file)
    print(len(data_test))
    for trial in data_test:
        print(trial['sign'].shape)
        break

file.close()

1276
torch.Size([165, 224])
